# Estimación de Carbono Aéreo — Parque Nacional Iguazú (Misiones, Argentina)
## Pipeline: GEDI L2A/L4A + Sentinel-2 L2A + Random Forest

**Autor:** Darío Nicolás Sánchez Leguizamón · UNAJ · sdarionicolas@gmail.com

Adaptación del pipeline de la Reserva de Biósfera Pereyra Iraola a la **Selva Paranaense**.

### Qué cambia respecto de Pereyra
| Aspecto | Pereyra | Iguazú (este notebook) |
|---|---|---|
| Área | rectángulo | **polígono real** del Parque (WDPA) |
| Máscara GEDI | calidad, sensibilidad, elevación | + **adquisición nocturna** + **filtro de pendiente** |
| Máscara Sentinel-2 | solo % de nubes de la escena | + **máscara por píxel (SCL)** |
| Unidad de entrenamiento | raster de carbono remuestreado a 10 m | **píxeles GEDI de 25 m** con predictores agregados a 25 m (sin remuestreo del objetivo) |
| Variable objetivo | ecuación sobre RH95 (origen no documentado) | **dos objetivos**: (A) GEDI L4A (AGBD) y (B) la ecuación de Pereyra aplicada a RH95, solo para comparar |
| Validación | división aleatoria de píxeles | **validación cruzada espacial por bloques** (+ aleatoria, para comparar) |
| Contraste externo | — | rangos de inventario (INBN2, parcelas de campo) y GEDI L4B |

### Alcance y límites (leer antes de interpretar)
- El resultado es una **estimación modelada**, no carbono medido. Sin parcelas georreferenciadas dentro del Parque, la validación es **preliminar**.
- El objetivo (A) es un producto modelado de la NASA, calibrado globalmente; el (B) hereda los supuestos de la ecuación de Pereyra, sin validar con parcelas.
- Las parcelas de campo conocidas (Abranchuk et al. 2018) tienen **coordenadas pendientes**: ver Módulo 7.

## Módulo 0 — Configuración

In [ ]:
# Ejecutar solo la primera vez en el entorno Colab
!pip install geemap geopandas rasterio matplotlib pandas scikit-learn --quiet

In [ ]:
# ============================================================
# PARÁMETROS GLOBALES — Editá solo esta sección
# ============================================================

# ── GEE ──────────────────────────────────────────────────────
GEE_PROJECT = 'TU_PROYECTO_GEE'            # Reemplazá con tu proyecto GEE

# ── Área de estudio ──────────────────────────────────────────
WDPA_ISO3          = 'ARG'
WDPA_NAME_CONTAINS = 'Iguaz'               # filtra por nombre en la base WDPA (sensible a mayúsculas)
WDPA_EXCLUIR       = ['Alto']              # excluye todo nombre que CONTENGA esto ("Alto Iguazu/Iguazú":
                                           # reserva distinta de 0,8 km², no forma parte del Parque)
# Solo se usa si WDPA no devuelve nada. Es un rectángulo APROXIMADO: revisalo en el mapa.
BBOX_FALLBACK = [-54.60, -25.95, -54.05, -25.45]   # lon_min, lat_min, lon_max, lat_max

# ── Google Drive ─────────────────────────────────────────────
DRIVE_FOLDER = 'IGUAZU'

# ── Sentinel-2 ───────────────────────────────────────────────
S2_ANIO       = '2023'
S2_NUBES_PCT  = 40                         # más laxo que Pereyra (10): la región es muy nubosa;
                                           # la máscara SCL por píxel limpia el resto
# ── GEDI ─────────────────────────────────────────────────────
GEDI_INICIO      = '2019-04-18'
GEDI_FIN         = '2023-12-31'
GEDI_SENSITIVITY = 0.95
GEDI_ELEV_DIFF   = 50                      # |elev GEDI − SRTM| máx. (m)
PENDIENTE_MAX    = 20                      # grados; GEDI sobreestima altura en laderas empinadas

# ── Modelo Random Forest ─────────────────────────────────────
RF_N_ESTIMATORS = 300
RF_MAX_DEPTH    = 20
RF_TEST_SIZE    = 0.30
RF_RANDOM_STATE = 42
CV_BLOQUE_M     = 2000                     # lado del bloque espacial (m)
CV_FOLDS        = 5

# ── Carbono ──────────────────────────────────────────────────
CARBON_FACTOR = 0.47                       # biomasa → carbono (Winrock, 2018; IPCC)
WOOD_DENSITY  = 0.6                        # solo para el objetivo B (ecuación de Pereyra)
NDVI_MIN_MAPA = 0.4                        # el mapa final se enmascara por debajo de este NDVI
                                           # (agua, suelo desnudo, saltos de agua)

In [ ]:
import os, time
import ee, geemap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import reproject
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, GroupKFold, cross_val_predict
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

ee.Authenticate()
ee.Initialize(project=GEE_PROJECT)

from google.colab import drive
drive.mount('/content/drive')

OUTPUT_DIR = f'/content/drive/MyDrive/{DRIVE_FOLDER}/'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'✅ Carpeta de salida: {OUTPUT_DIR}')

## Módulo 1 — Área de estudio
Se toma el polígono del Parque desde la base mundial de áreas protegidas (WDPA) en GEE.
**Revisá la lista impresa**: si aparece más de una unidad (p. ej. Parque y Reserva Nacional), decidí cuál corresponde.

In [ ]:
wdpa = (ee.FeatureCollection('WCMC/WDPA/current/polygons')
        .filter(ee.Filter.eq('ISO3', WDPA_ISO3))
        .filter(ee.Filter.stringContains('NAME', WDPA_NAME_CONTAINS))
        .map(lambda f: f.set('AREA_POLIGONO_KM2', f.geometry().area(100).divide(1e6))))

for texto in WDPA_EXCLUIR:
    wdpa = wdpa.filter(ee.Filter.stringContains('NAME', texto).Not())

encontrados = wdpa.size().getInfo()
if encontrados > 0:
    feats = wdpa.getInfo()['features']
    print('Propiedades disponibles:', sorted(feats[0]['properties'].keys()))
    print('\nPolígonos seleccionados (área reportada vs. área real del polígono):')
    for f in feats:
        p = f['properties']
        print(f"  • {p.get('NAME')} | {p.get('DESIG_ENG')} | reportada {p.get('REP_AREA')} km² "
              f"| polígono {p.get('AREA_POLIGONO_KM2'):.1f} km² | GIS_AREA {p.get('GIS_AREA')}")
    roi = wdpa.geometry().dissolve(100)
else:
    print('⚠️  WDPA no devolvió resultados: se usa el rectángulo aproximado BBOX_FALLBACK.')
    roi = ee.Geometry.Rectangle(BBOX_FALLBACK)

area_km2 = roi.area(100).divide(1e6).getInfo()
print(f'\nÁrea de estudio (unión): {area_km2:,.0f} km²')
print('Referencia: Parque (599,45) + Reserva Nacional (76,75) reportados ≈ 676 km²; el polígono de OSM suma 582 km² y el de WDPA 591 km².')
print('Si la unión da bastante menos, el polígono WDPA está incompleto: revisá el mapa antes de seguir.')

Map = geemap.Map()
Map.centerObject(roi, 10)
Map.addLayer(wdpa.style(color='yellow', fillColor='00000000', width=2), {}, 'Polígonos WDPA')
Map.addLayer(roi.bounds(), {'color': 'red'}, 'Extensión de exportación')
Map

## Módulo 2 — GEDI L2A (altura) y L4A (biomasa) + exportación
Filtros: `quality_flag`=1, `degrade_flag`=0, sensibilidad ≥ umbral, **adquisición nocturna** (`solar_elevation` < 0),
diferencia de elevación GEDI–SRTM ≤ umbral y **pendiente ≤ PENDIENTE_MAX**.
Se exporta un raster de 25 m con dos bandas: `RH95` (m) y `AGBD` (Mg/ha de biomasa, L4A).

> ⚠️ Los nombres de colección y bandas corresponden al catálogo GEE a la fecha de redacción. Si GEE devuelve un error de banda inexistente, revisá el catálogo (`LARSE/GEDI/GEDI02_A_002_MONTHLY`, `LARSE/GEDI/GEDI04_A_002_MONTHLY`).

In [ ]:
srtm      = ee.Image('USGS/SRTMGL1_003').select('elevation')
pendiente = ee.Terrain.slope(srtm)

def filtrar_l2a(img):
    m = (img.select('quality_flag').eq(1)
         .And(img.select('degrade_flag').eq(0))
         .And(img.select('sensitivity').gte(GEDI_SENSITIVITY))
         .And(img.select('solar_elevation').lt(0))
         .And(img.select('elev_lowestmode').subtract(srtm).abs().lte(GEDI_ELEV_DIFF))
         .And(pendiente.lte(PENDIENTE_MAX)))
    return img.updateMask(m)

def filtrar_l4a(img):
    m = (img.select('l4_quality_flag').eq(1)
         .And(img.select('degrade_flag').eq(0))
         .And(img.select('sensitivity').gte(GEDI_SENSITIVITY))
         .And(img.select('solar_elevation').lt(0))
         .And(pendiente.lte(PENDIENTE_MAX)))
    return img.updateMask(m)

l2a = (ee.ImageCollection('LARSE/GEDI/GEDI02_A_002_MONTHLY')
       .filterBounds(roi).filterDate(GEDI_INICIO, GEDI_FIN).map(filtrar_l2a))
l4a = (ee.ImageCollection('LARSE/GEDI/GEDI04_A_002_MONTHLY')
       .filterBounds(roi).filterDate(GEDI_INICIO, GEDI_FIN).map(filtrar_l4a))

rh95 = l2a.select('rh95').median().rename('RH95')
agbd = l4a.select('agbd').median().rename('AGBD')
gedi_img = rh95.addBands(agbd).clip(roi).toFloat()

task_gedi = ee.batch.Export.image.toDrive(
    image=gedi_img, description='GEDI_Iguazu_25m', folder=DRIVE_FOLDER,
    fileNamePrefix='GEDI_Iguazu_25m', region=roi.bounds(), scale=25,
    crs='EPSG:4326', maxPixels=1e13)
task_gedi.start()
print('🚀 Exportación GEDI iniciada → GEDI_Iguazu_25m.tif (bandas: RH95, AGBD)')

## Módulo 3 — Sentinel-2 L2A: mosaico anual y stack de 8 capas
Máscara por píxel con la banda **SCL** (sombra, nubes, cirros, nieve). Mediana anual. Las 8 capas se **agregan a 25 m (media)**
para coincidir con el pie de huella de GEDI. Son las mismas 8 variables que en Pereyra, para poder comparar la importancia de B11.

In [ ]:
def mascara_scl(img):
    scl = img.select('SCL')
    ok = (scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10)).And(scl.neq(11)))
    return img.updateMask(ok)

s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
      .filterBounds(roi)
      .filterDate(f'{S2_ANIO}-01-01', f'{S2_ANIO}-12-31')
      .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', S2_NUBES_PCT))
      .map(mascara_scl))
print('Escenas S2 utilizadas:', s2.size().getInfo())

BANDAS = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12']
NOMBRES_FEATURES = BANDAS + ['NDVI', 'NDWI']

proj10  = s2.first().select('B2').projection()
mos     = s2.median().select(BANDAS)
ndvi    = mos.normalizedDifference(['B8', 'B4']).rename('NDVI')
ndwi    = mos.normalizedDifference(['B3', 'B11']).rename('NDWI')
stack10 = mos.addBands([ndvi, ndwi]).toFloat()

stack25 = (stack10.setDefaultProjection(proj10)
           .reduceResolution(ee.Reducer.mean(), maxPixels=1024)
           .reproject(crs='EPSG:4326', scale=25)
           .clip(roi))

task_s2 = ee.batch.Export.image.toDrive(
    image=stack25, description='Sentinel2_Iguazu_25m', folder=DRIVE_FOLDER,
    fileNamePrefix='Sentinel2_Iguazu_25m', region=roi.bounds(), scale=25,
    crs='EPSG:4326', maxPixels=1e13)
task_s2.start()
print('🚀 Exportación Sentinel-2 iniciada → Sentinel2_Iguazu_25m.tif (8 bandas, 25 m)')

In [ ]:
# Esperar a que terminen las dos exportaciones (puede tardar varios minutos)
for t in (task_gedi, task_s2):
    while t.active():
        print('⏳', t.status()['description'], t.status()['state'])
        time.sleep(30)
    st = t.status()
    print('✔' if st['state'] == 'COMPLETED' else '✖', st['description'], st['state'], st.get('error_message', ''))

## Módulo 4 — Variables objetivo de carbono (tC/ha)
- **A · `L4A`**: AGBD del producto GEDI L4A × factor de carbono. Es un modelo calibrado globalmente con parcelas de campo.
- **B · `EcuacionRH95`**: la ecuación que generó los datos de Pereyra, `C = 0,0673·(0,6·h)^1,23·100·0,47` (tC/ha), aplicada a RH95.
  ⚠️ Su origen no está documentado ni validado con parcelas, y frente a GEDI L4A sobreestima la biomasa 2 a 3 veces en alturas
  menores de 15 m (ver `docs/CORRECCIONES_PEREYRA.md`, §3.10). Se incluye **solo para comparabilidad**; el resultado principal es el (A).

In [ ]:
def rh95_a_carbono_ecuacion(h, rho=WOOD_DENSITY, cf=CARBON_FACTOR):
    # Ecuación efectivamente usada en el trabajo de Pereyra (tC/ha). Sin fuente documentada ni validación de campo.
    return 0.0673 * (rho * np.maximum(h, 0)) ** 1.23 * 100 * cf

with rasterio.open(OUTPUT_DIR + 'GEDI_Iguazu_25m.tif') as g:
    gedi = g.read().astype('float32')           # [RH95, AGBD]
    gprof, gT, gcrs = g.profile.copy(), g.transform, g.crs
    H, W = g.height, g.width

rh, agbd_arr = gedi[0], gedi[1]
for a in (rh, agbd_arr):
    a[(a <= 0) | ~np.isfinite(a)] = np.nan

OBJETIVOS = {
    'L4A':     agbd_arr * CARBON_FACTOR,
    'EcuacionRH95': rh95_a_carbono_ecuacion(rh),
}

# Stack Sentinel-2 llevado exactamente a la grilla GEDI
with rasterio.open(OUTPUT_DIR + 'Sentinel2_Iguazu_25m.tif') as s:
    s2_arr = np.full((s.count, H, W), np.nan, dtype='float32')
    for i in range(s.count):
        reproject(source=rasterio.band(s, i + 1), destination=s2_arr[i],
                  dst_transform=gT, dst_crs=gcrs, resampling=Resampling.average)

feats_ok = np.all(np.isfinite(s2_arr), axis=0) & (s2_arr[3] > 0)    # B8 > 0

# Coordenadas de cada píxel y área en ha (la grilla es geográfica: el área varía con la latitud)
cols, rows = np.meshgrid(np.arange(W), np.arange(H))
XS = gT.c + gT.a * (cols + 0.5)
YS = gT.f + gT.e * (rows + 0.5)
lat_f = gT.f + gT.e * (np.arange(H) + 0.5)
area_ha_fila = (abs(gT.a) * 111320 * np.cos(np.radians(lat_f))) * (abs(gT.e) * 110540) / 1e4
AREA_HA = np.repeat(area_ha_fila[:, None], W, axis=1)

print(f'Grilla: {H} × {W} píxeles | píxeles con S2 válido: {feats_ok.sum():,}')
for k, v in OBJETIVOS.items():
    n = int((feats_ok & np.isfinite(v)).sum())
    print(f'  Objetivo {k:8s}: {n:,} huellas GEDI utilizables | media {np.nanmean(v):.1f} tC/ha')

## Módulo 5 — Random Forest: validación aleatoria vs. validación espacial
Se reportan **ambas**: la división aleatoria (como en Pereyra, optimista por autocorrelación espacial) y la
**validación cruzada por bloques** de `CV_BLOQUE_M` metros, que es la medida que corresponde citar.

In [ ]:
def grupos_bloques(xs, ys, lado_m):
    dx = lado_m / (111320 * np.cos(np.radians(np.mean(ys))))
    dy = lado_m / 110540
    return (np.floor((xs - xs.min()) / dx) * 100000 + np.floor((ys - ys.min()) / dy)).astype(int)

def metricas(y, p):
    return dict(R2=r2_score(y, p), RMSE=float(np.sqrt(mean_squared_error(y, p))), MAE=mean_absolute_error(y, p))

def nuevo_rf():
    return RandomForestRegressor(n_estimators=RF_N_ESTIMATORS, max_depth=RF_MAX_DEPTH,
                                 random_state=RF_RANDOM_STATE, n_jobs=-1)

modelos, oof, filas = {}, {}, []

for nombre, carbono in OBJETIVOS.items():
    m  = feats_ok & np.isfinite(carbono) & (carbono > 0)
    X  = s2_arr[:, m].T
    y  = carbono[m]
    gr = grupos_bloques(XS[m], YS[m], CV_BLOQUE_M)

    # (1) división aleatoria — comparable con Pereyra
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=RF_TEST_SIZE, random_state=RF_RANDOM_STATE)
    r_alea = metricas(yte, nuevo_rf().fit(Xtr, ytr).predict(Xte))

    # (2) validación cruzada espacial por bloques
    p_cv = cross_val_predict(nuevo_rf(), X, y, groups=gr, cv=GroupKFold(n_splits=CV_FOLDS))
    r_esp = metricas(y, p_cv)
    oof[nombre] = (y, p_cv)

    # (3) modelo final con todas las muestras
    modelos[nombre] = nuevo_rf().fit(X, y)

    filas.append({'Objetivo': nombre, 'N': len(y),
                  'R2 aleatorio': r_alea['R2'], 'RMSE aleatorio': r_alea['RMSE'],
                  'R2 espacial': r_esp['R2'], 'RMSE espacial': r_esp['RMSE'], 'MAE espacial': r_esp['MAE']})

df_met = pd.DataFrame(filas).round(3)
print(df_met.to_string(index=False))
df_met.to_csv(OUTPUT_DIR + 'Metricas_Iguazu.csv', index=False)
print('\nSi R² aleatorio ≫ R² espacial, la diferencia es autocorrelación espacial, no capacidad predictiva real.')

## Módulo 6 — Mapas, importancia de variables y stock

In [ ]:
mapas, resumen = {}, []
for nombre, rf in modelos.items():
    mapa = np.full((H, W), np.nan, dtype='float32')
    ok = feats_ok & (s2_arr[6] >= NDVI_MIN_MAPA)                  # banda 6 = NDVI
    mapa[ok] = rf.predict(s2_arr[:, ok].T)
    mapas[nombre] = mapa

    prof = gprof.copy(); prof.update(count=1, dtype='float32', nodata=np.nan)
    with rasterio.open(OUTPUT_DIR + f'Carbono_Predicho_{nombre}.tif', 'w', **prof) as dst:
        dst.write(mapa, 1)

    imp = pd.Series(rf.feature_importances_, index=NOMBRES_FEATURES).sort_values(ascending=False)
    stock_ktc = np.nansum(mapa * AREA_HA) / 1e3
    resumen.append({'Objetivo': nombre, 'Media (tC/ha)': np.nanmean(mapa), 'Stock (ktC)': stock_ktc,
                    'Variable #1': imp.index[0], 'Importancia #1': imp.iloc[0],
                    'Imp. NDVI': imp['NDVI'], 'Imp. B11': imp['B11']})

    fig, ax = plt.subplots(1, 3, figsize=(20, 6))
    y, p = oof[nombre]
    ax[0].scatter(y, p, s=6, alpha=0.2); lim = [0, np.nanpercentile(y, 99.5)]
    ax[0].plot(lim, lim, 'r--'); ax[0].set_xlim(lim); ax[0].set_ylim(lim)
    ax[0].set_xlabel('Observado (tC/ha)'); ax[0].set_ylabel('Predicho (tC/ha)')
    ax[0].set_title(f'{nombre} — validación espacial (bloques {CV_BLOQUE_M} m)')
    im = ax[1].imshow(mapa, cmap='RdYlGn', vmin=np.nanpercentile(mapa, 2), vmax=np.nanpercentile(mapa, 98))
    plt.colorbar(im, ax=ax[1], label='tC/ha'); ax[1].axis('off'); ax[1].set_title(f'Carbono aéreo predicho — {nombre}')
    imp.sort_values().plot.barh(ax=ax[2], color=['#2ecc71' if v == 'B11' else '#3498db' for v in imp.sort_values().index])
    ax[2].set_title('Importancia de variables (Gini)')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR + f'Resumen_{nombre}.png', dpi=200, bbox_inches='tight'); plt.show()

df_res = pd.DataFrame(resumen).round(3)
print(df_res.to_string(index=False))
df_res.to_csv(OUTPUT_DIR + 'Resumen_Iguazu.csv', index=False)

## Módulo 6b — Lo que sí se sostiene con huellas GEDI
En la primera corrida, Sentinel-2 + RF **no predijo el carbono L4A píxel a píxel** (R² espacial ≈ 0,08), ni limpiando outliers.
Dos resultados sí son defendibles:
1. **Estimación por muestreo** de la media del Parque directamente desde las huellas GEDI, con intervalo de confianza por
   *bootstrap de bloques* (respeta la autocorrelación espacial). No depende de Sentinel-2 ni del RF.
2. **Predicción de la altura** (RH95), donde Sentinel-2 sí muestra algo de señal (R² espacial ≈ 0,2).

> El supuesto del punto 1 es que las huellas filtradas representan al Parque (cobertura sistemática de las órbitas GEDI,
> pero con filtros de pendiente, noche y calidad). No es un diseño probabilístico.

In [ ]:
def media_bootstrap_bloques(valores, xs, ys, lado_m=CV_BLOQUE_M, n_boot=2000, semilla=1):
    # Media e IC95 % remuestreando bloques espaciales completos (no huellas sueltas)
    g = grupos_bloques(xs, ys, lado_m)
    _, inv = np.unique(g, return_inverse=True)
    suma, n = np.bincount(inv, valores), np.bincount(inv)
    k = np.random.default_rng(semilla).integers(0, len(n), size=(n_boot, len(n)))
    est = suma[k].sum(axis=1) / n[k].sum(axis=1)
    return valores.mean(), np.percentile(est, [2.5, 97.5]), len(n)

filas = []
for tope in (400, 1000):                                   # sensibilidad al tope de AGBD (Mg/ha)
    m = np.isfinite(agbd_arr) & (agbd_arr <= tope)
    media, (lo, hi), nb = media_bootstrap_bloques(agbd_arr[m] * CARBON_FACTOR, XS[m], YS[m])
    filas.append({'AGBD máx (Mg/ha)': tope, 'N huellas': int(m.sum()), 'Bloques': nb,
                  'Media (tC/ha)': media, 'IC95 inf': lo, 'IC95 sup': hi,
                  'Mediana (tC/ha)': float(np.median(agbd_arr[m] * CARBON_FACTOR))})
df_muestreo = pd.DataFrame(filas).round(1)
print(df_muestreo.to_string(index=False))
df_muestreo.to_csv(OUTPUT_DIR + 'Estimacion_por_huellas_Iguazu.csv', index=False)

In [ ]:
# Predicción de la altura RH95 (m) con Sentinel-2: validación espacial por bloques
m = feats_ok & np.isfinite(rh) & (rh >= 5) & (rh <= 50) & (s2_arr[6] >= 0.6)    # banda 6 = NDVI
X, y = s2_arr[:, m].T, rh[m]
gr = grupos_bloques(XS[m], YS[m], CV_BLOQUE_M)
p = cross_val_predict(nuevo_rf(), X, y, groups=gr, cv=GroupKFold(n_splits=CV_FOLDS))
print(f'RH95 ← Sentinel-2 | N={len(y):,} | R² espacial = {r2_score(y, p):.3f} | '
      f'RMSE = {np.sqrt(mean_squared_error(y, p)):.2f} m | MAE = {mean_absolute_error(y, p):.2f} m')

## Módulo 6c — Altura de dosel de Meta/WRI (1 m) frente a GEDI
El mapa de altura de Meta/WRI (Tolan et al. 2024) es **denso** (1 m) y se obtuvo con visión por computadora sobre imágenes de muy alta
resolución, calibrado con lidar. Es el insumo que se usó en Pereyra. Acá se usa de dos formas:
1. **Chequeo independiente**: ¿la altura de Meta coincide con RH95 de GEDI en las huellas del Parque?
2. **Predictor denso** del AGBD de GEDI L4A, como alternativa a Sentinel-2 (que se satura en selva cerrada).

> La banda `cover_code` es la del asset oficial de Meta en GEE (la misma que traía el raster de Pereyra).
> Se agregan 2 estadísticos a 25 m: media y percentil 95 de la altura de 1 m dentro de cada píxel.

In [ ]:
meta_col = ee.ImageCollection('projects/meta-forest-monitoring-okw37/assets/CanopyHeight')
meta1m   = meta_col.mosaic().select('cover_code')
proj1m   = meta_col.first().select('cover_code').projection()

reductor = ee.Reducer.mean().combine(ee.Reducer.percentile([95]), sharedInputs=True)
meta25 = (meta1m.setDefaultProjection(proj1m)
          .reduceResolution(reductor, maxPixels=1024)
          .reproject(crs='EPSG:4326', scale=25)
          .select(['cover_code_mean', 'cover_code_p95'], ['H_MEAN', 'H_P95'])
          .clip(roi).toFloat())

task_meta = ee.batch.Export.image.toDrive(
    image=meta25, description='Meta_Altura_Iguazu_25m', folder=DRIVE_FOLDER,
    fileNamePrefix='Meta_Altura_Iguazu_25m', region=roi.bounds(), scale=25,
    crs='EPSG:4326', maxPixels=1e13)
task_meta.start()
while task_meta.active():
    print('⏳', task_meta.status()['state']); time.sleep(30)
print('✔' if task_meta.status()['state'] == 'COMPLETED' else '✖', task_meta.status())

In [ ]:
with rasterio.open(OUTPUT_DIR + 'Meta_Altura_Iguazu_25m.tif') as s:
    meta_arr = np.full((s.count, H, W), np.nan, dtype='float32')
    for i in range(s.count):
        reproject(source=rasterio.band(s, i + 1), destination=meta_arr[i], dst_transform=gT,
                  dst_crs=gcrs, resampling=Resampling.nearest)
h_mean, h_p95 = meta_arr

# (1) ¿Meta coincide con GEDI RH95?
m = np.isfinite(h_p95) & np.isfinite(rh) & (rh <= 50)
sesgo = float(np.mean(h_p95[m] - rh[m]))
print(f'Meta H_P95 vs GEDI RH95 | N={m.sum():,} | r={np.corrcoef(h_p95[m], rh[m])[0,1]:.3f} | '
      f'RMSE={np.sqrt(np.mean((h_p95[m]-rh[m])**2)):.2f} m | sesgo (Meta−GEDI)={sesgo:+.2f} m')

# (2) AGBD de GEDI L4A predicho con la altura de Meta (con y sin Sentinel-2), validación espacial
m2 = np.isfinite(h_mean) & np.isfinite(h_p95) & np.isfinite(agbd_arr) & (agbd_arr <= 1000) & feats_ok
y  = agbd_arr[m2] * CARBON_FACTOR
gr = grupos_bloques(XS[m2], YS[m2], CV_BLOQUE_M)
for nombre, X in {'Meta (H_MEAN, H_P95)':        np.column_stack([h_mean[m2], h_p95[m2]]),
                  'Sentinel-2 (8 bandas)':        s2_arr[:, m2].T,
                  'Meta + Sentinel-2':            np.column_stack([h_mean[m2], h_p95[m2], s2_arr[:, m2].T])}.items():
    p = cross_val_predict(nuevo_rf(), X, y, groups=gr, cv=GroupKFold(n_splits=CV_FOLDS))
    print(f'AGBD L4A ← {nombre:24s} | N={len(y):,} | R² espacial = {r2_score(y, p):.3f} | RMSE = {np.sqrt(mean_squared_error(y, p)):.1f} tC/ha')

## Conclusiones de la primera corrida (Parque Nacional Iguazú, polígono WDPA de 591 km²)

| Pregunta | Resultado |
|---|---|
| ¿Cuánto carbono tiene el Parque? | **41,5 tC/ha** (IC95 % 40,2 – 42,7; estimación por muestreo de 24.603 huellas GEDI L4A, bootstrap por bloques de 2 km). GEDI L4B: 43,8 tC/ha |
| ¿Se puede mapear píxel a píxel con Sentinel-2 + Random Forest? | **No.** R² espacial ≈ 0,08 (0,10 con la altura de Meta/WRI agregada); limpiar outliers casi no cambia el resultado. El mapa muestra cauces y claros, no carbono |
| ¿Y la altura del dosel? | Sentinel-2 explica algo: R² espacial ≈ 0,21 (RMSE 3,9 m) |
| ¿Mejora a escalas más gruesas? | Sí, hasta ≈ 0,21 a 1 km; el ruido por huella se promedia, pero la señal sigue siendo débil |
| ¿Meta/WRI coincide con GEDI en Iguazú? | r = 0,47; sesgo −0,6 m (RMSE 4,4 m) |
| ¿Comparación con Pereyra (mismo producto GEDI L4A)? | Iguazú tiene ≈ **3,6 veces más** carbono por hectárea (41,5 vs. 11,5 tC/ha) |

**Qué se puede afirmar:** la media del Parque por muestreo de huellas, con su intervalo; la estructura del dosel (RH95 medio 18 m, 99 % de las huellas ≥ 5 m).
**Qué no:** un mapa de carbono ni valores absolutos validados. La media GEDI es menor que las referencias disponibles (~60–125 tC/ha), pero la comparación no es pareja: las parcelas del Parque se ubicaron en bosque de dosel cerrado (DAP ≥ 5 cm), y la media GEDI abarca todo el Parque, con zonas abiertas. Solo la validación con parcelas de campo puede decidir si hay un sesgo del producto o una diferencia real del bosque.
**Límite del Parque:** el polígono de WDPA (591 km²) y el de OpenStreetMap (582 km²; `iguazu_osm.geojson`) dan la misma media (L4A 41,5 tC/ha). El área declarada (≈ 676 km²) es mayor y no se sabe cuál es la geometría legal: el stock total depende de ello, la media por hectárea no.
**Sobre el método de Pereyra:** su R² alto proviene de reproducir un carbono derivado de la altura de Meta; con un objetivo independiente (GEDI) el R² colapsa. Ver `CORRECCIONES_PEREYRA.md`.

## Módulo 7 — Contraste con información externa (validación preliminar)
**a) Rangos de referencia publicados** (no son verdad de campo del Parque).
**b) Parcelas de campo dentro del Parque**: El Cruce (BN4) y Sendero Macuco (BN5), de Abranchuk et al. (2018).
Sus coordenadas son de la autora de la tesis y **no se publican en este repositorio**; el uso de esos datos se acordará con sus propietarias (IBS-CONICET).
**c) GEDI L4B** (grilla de 1 km, producto independiente de nuestro RF).

In [ ]:
REFERENCIAS_TC_HA = {
    'INBN2 Selva Paranaense (media regional, LEDS-LAC)':        59.7,
    'Abranchuk et al. 2018 — promedio 7 parcelas NO Misiones':  98.97,
    'Abranchuk et al. 2018 — BN4 El Cruce, PN Iguazú':          68.24,
    'Abranchuk et al. 2018 — BN5 Sendero Macuco, PN Iguazú':    67.55,
    'Fonsêca et al. 2024 — bosque maduro, Bosque Atlántico':    125.34,
}
print('Media del mapa por objetivo (tC/ha):', {k: round(float(np.nanmean(v)), 1) for k, v in mapas.items()})
print('\nReferencias:')
for k, v in REFERENCIAS_TC_HA.items():
    print(f'  {v:7.1f}  {k}')

# ── Parcelas de campo (completar coordenadas en grados decimales, WGS84) ──
PARCELAS_CAMPO = {
    'BN4 El Cruce':      {'lon': None, 'lat': None, 'C_obs': 68.24},
    'BN5 Sendero Macuco': {'lon': None, 'lat': None, 'C_obs': 67.55},
}
for nombre, p in PARCELAS_CAMPO.items():
    if p['lon'] is None:
        print(f'\n⏳ {nombre}: sin coordenadas — comparación pendiente.')
        continue
    r, c = rasterio.transform.rowcol(gT, p['lon'], p['lat'])
    for obj, mapa in mapas.items():
        v = np.nanmean(mapa[max(r - 1, 0):r + 2, max(c - 1, 0):c + 2])      # ventana 3×3 (≈75 m)
        print(f'{nombre} | {obj}: predicho {v:.1f} vs observado {p["C_obs"]:.1f} tC/ha')

# ── GEDI L4B (1 km) como chequeo cruzado ──
try:
    l4b = ee.Image('LARSE/GEDI/GEDI04_B_002').select('MU')
    mu = l4b.reduceRegion(ee.Reducer.mean(), roi, 1000, maxPixels=1e9).getInfo().get('MU')
    print(f'\nGEDI L4B — AGBD medio en el Parque: {mu:.1f} Mg/ha ≈ {mu * CARBON_FACTOR:.1f} tC/ha')
except Exception as e:
    print('\nGEDI L4B no disponible:', e)

## Verificación final

In [ ]:
esperados = ['GEDI_Iguazu_25m.tif', 'Sentinel2_Iguazu_25m.tif',
             'Carbono_Predicho_L4A.tif', 'Carbono_Predicho_EcuacionRH95.tif',
             'Resumen_L4A.png', 'Resumen_EcuacionRH95.png', 'Metricas_Iguazu.csv', 'Resumen_Iguazu.csv']
for f in esperados:
    p = OUTPUT_DIR + f
    print(('  ✅' if os.path.exists(p) else '  ❌ FALTANTE'), f, f'({os.path.getsize(p)/1e6:.1f} MB)' if os.path.exists(p) else '')

## Referencias
- Abranchuk, M. F. (2018). *Almacenamiento y tiempo medio de residencia del carbono en la biomasa aérea del bosque nativo y plantaciones de* Pinus taeda *en el NO de Misiones.* UNaM.
- Chave, J., et al. (2005; 2014). Allometric models for tropical trees / Improved allometric models. *Oecologia; Global Change Biology.*
- Dubayah, R., et al. (2020, 2022). GEDI mission; GEDI L4A footprint-level biomass. *Science of Remote Sensing; ORNL DAAC.*
- Fonsêca, N., et al. (2024). Carbon stock in aboveground biomass and necromass in the Atlantic Forest. *An. Acad. Bras. Ciênc.*
- Dirección Nacional de Bosques (2021). *Segundo Inventario Nacional de Bosques Nativos — Informe regional Selva Paranaense.* MAyDS.
- Bullock, E. L., et al. (2023). Estimating aboveground biomass density using hybrid statistical inference with GEDI lidar data and Paraguay's national forest inventory. *Environ. Res. Lett.*
- Winrock International (2018). *Calculating carbon stocks: a guidance module.*